# Symbolic FPJ parity-game solver

This notebook lets you select any SYNTCOMP benchmark by name. It visualizes the selected game first, runs FPJ, then finds all rotation-unique cycles and checks which cycles are compatible with a strategy that guarantees the winner. The small project examples and image game remain available for comparison.

The solver represents vertex sets and the edge relation with BDDs from `dd.autoref`. Priorities use the max-parity convention. The implementation follows Algorithm 2 in [Lijzenga and van Dijk, *Symbolic Parity Game Solvers that Yield Winning Strategies*](https://ris.utwente.nl/ws/portalfiles/portal/253278668/Lijzenga2020symbolic.pdf).

In [ ]:
from pathlib import Path
import sys

SYNTCOMP_GAME = "lilydemo03"

# Jupyter may start in the repository root, code/, or notebooks/.
search_roots = (Path.cwd(), *Path.cwd().parents)
CODE = next(
    (candidate for root in search_roots
     for candidate in (root, root / 'code')
     if (candidate / 'src' / 'parity_bdd_lab').is_dir()),
    None,
)
if CODE is None:
    raise FileNotFoundError(f'Could not locate the code project from {Path.cwd()}')
sys.path.insert(0, str(CODE / 'src'))

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
from parity_bdd_lab import draw_game, enumerate_forced_winning_cycles, enumerate_unique_cycles, fpj, read_pg
from parity_bdd_lab.examples import tiny_games

image_labels = dict(enumerate('abcdefghi'))
image_game = read_pg(CODE / 'data' / 'games' / 'image_example.pg', vertex_labels=image_labels)
games = (*tiny_games(), image_game)
benchmark_file = CODE / 'data' / 'benchmarks' / 'syntcomp2020-pgame' / f"{SYNTCOMP_GAME}.tlsf.ehoa.oink"
game = read_pg(benchmark_file, name=SYNTCOMP_GAME)
[(game.name, len(game.vertices)) for game in games] + [(game.name, len(game.vertices))]

## 1. Visualize the game

This is the first analysis step. Vertices are labelled with their game IDs and priorities; color and shape indicate ownership.

In [ ]:
draw_game(game, layout='spring', title=f"{SYNTCOMP_GAME} parity game")

## 2. Run symbolic FPJ

FPJ uses BDDs from `dd` to compute winning regions and strategy edges.

In [ ]:
fpj_result = fpj(game)
print('Even winning region:', tuple(sorted(fpj_result.even_wins)))
print('Odd winning region:', tuple(sorted(fpj_result.odd_wins)))
print('Even winning strategy edges:', tuple(sorted(fpj_result.even_strategy)))
print('Odd winning strategy edges:', tuple(sorted(fpj_result.odd_strategy)))
print(f'FPJ iterations: {fpj_result.iterations}; BDD nodes: {fpj_result.bdd_nodes}')
assert fpj_result.even_wins.isdisjoint(fpj_result.odd_wins)
assert fpj_result.even_wins | fpj_result.odd_wins == set(game.vertices)

## 3. Enumerate rotation-unique cycles

The utility enumerates the graph’s directed simple cycles directly using Johnson’s algorithm. Every possible eventual cycle from a positional profile is one of these cycles, and each simple cycle can be realized by choosing its edges at its vertices. Cycles are returned once up to rotation; reverse directions remain distinct.

In [ ]:
cycles = enumerate_unique_cycles(game)
cycle_rows = [{
    'cycle': cycle.vertices,
    'cycle length': len(cycle.vertices),
    'cycle priorities': cycle.priorities,
    'greatest recurring priority': cycle.greatest_recurring_priority,
    'winner': f'Player {cycle.winner}',
} for cycle in cycles]
unique_cycles = pd.DataFrame(cycle_rows)
display(unique_cycles)


## 4. Keep cycles compatible with a forced win

For each cycle, the utility restricts the cycle winner's choices at their owned cycle vertices to the cycle edges and solves the restricted game. It reports cycles for which that player can still guarantee a win from a cycle vertex. This avoids enumerating the positional strategy profiles.

In [ ]:
forced_cycles = enumerate_forced_winning_cycles(game)
forced_rows = [{
    'cycle': item.cycle.vertices,
    'cycle priorities': item.cycle.priorities,
    'winner who can force a win': f'Player {item.cycle.winner}',
    'winning from cycle vertices': item.starts,
    'winning strategy edges': item.winning_strategy,
} for item in forced_cycles]
forced_winning_cycles = pd.DataFrame(forced_rows)
display(forced_winning_cycles)


## Run FPJ on every example

Each strategy is returned as a set of winning edges. Vertices controlled by the other player are not included in that player's strategy. The symbolic strategy can retain multiple winning edges from a vertex when each is valid.

In [ ]:
def ordered_edges(edges):
    return tuple(sorted(edges, key=lambda edge: tuple(map(str, edge))))

results = {}
rows = []
for example_game in games:
    result = fpj(example_game)
    results[example_game.name] = result
    assert result.even_wins.isdisjoint(result.odd_wins)
    assert result.even_wins | result.odd_wins == set(example_game.vertices)
    rows.append({
        'game': example_game.name,
        'Even wins': tuple(sorted(result.even_wins, key=str)),
        'Odd wins': tuple(sorted(result.odd_wins, key=str)),
        'Even strategy edges': ordered_edges(result.even_strategy),
        'Odd strategy edges': ordered_edges(result.odd_strategy),
        'FPJ iterations': result.iterations,
        'BDD nodes': result.bdd_nodes,
    })

summary = pd.DataFrame(rows)
display(summary)

## Inspect the image example

For the image game, the output lists every vertex in the winning regions and the strategy edges FPJ selected. An edge `('d', 'e')`, for example, means the strategy chooses `d → e`.

In [ ]:
image_result = results['image_example']
print('Player 0 winning vertices:', sorted(image_result.even_wins))
print('Player 1 winning vertices:', sorted(image_result.odd_wins))
print('Player 0 winning strategy:', ordered_edges(image_result.even_strategy))
print('Player 1 winning strategy:', ordered_edges(image_result.odd_strategy))

## Solver output

`FPJResult` contains `even_wins`, `odd_wins`, `even_strategy`, `odd_strategy`, all justification edges, the number of iterations, and the BDD node count. The notebook checks that the two winning regions form a partition of each game.